### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sat Jun 27 05:32:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   71C    P0            219W /  300W |   29873MiB /  81920MiB |     76%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [9]:
max_iter=30000

## Extra Ablation

### PG EM - flipped orders

In [10]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [11]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  2%|█▏                                                                                 | 3/200 [00:00<00:26,  7.55it/s]

Iter 000: Log-likelihood = -0.349581


 38%|███████████████████████████████▏                                                  | 76/200 [00:06<00:09, 12.42it/s]


Converged at iter 76, ΔLL = 9.536743e-07
cuda


...

Converged at iter 76, ΔLL = 9.834766e-07
cuda


  2%|█▏                                                                                 | 3/200 [00:00<00:09, 20.12it/s]

Iter 000: Log-likelihood = -0.349576


 38%|███████████████████████████████▏                                                  | 76/200 [00:04<00:07, 16.86it/s]

Converged at iter 76, ΔLL = 9.834766e-07


In [12]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

5.53198652267456 +- 0.6764318648026948


In [13]:
PGEM_accs

[0.7921265959739685,
 0.7921266555786133,
 0.7921266555786133,
 0.7921265363693237,
 0.7921265959739685,
 0.7921266555786133,
 0.7921266555786133,
 0.7921267747879028,
 0.7921266555786133,
 0.7921265959739685]

In [14]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921266376972198 ± 5.990192664337688e-08, Weighted Accuracy: 0.8736116707324981 ± 7.077659420393656e-08, Kendall's Tau: 0.5794762463617944 ± 1.4268981930506516e-07


### PG EM - flipped orders (r then beta)

In [15]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [16]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  2%|█▋                                                                                 | 4/200 [00:00<00:10, 19.36it/s]

Iter 000: Log-likelihood = -0.349581


 38%|███████████████████████████████▏                                                  | 76/200 [00:04<00:07, 16.83it/s]


Converged at iter 76, ΔLL = 9.834766e-07
cuda


...

Converged at iter 76, ΔLL = 9.834766e-07
cuda


  2%|█▋                                                                                 | 4/200 [00:00<00:10, 19.14it/s]

Iter 000: Log-likelihood = -0.349576


 38%|███████████████████████████████▏                                                  | 76/200 [00:05<00:08, 14.44it/s]

Converged at iter 76, ΔLL = 9.834766e-07


In [17]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

5.966504383087158 +- 0.4236750854042795


In [18]:
PGEM_accs

[0.7921266555786133,
 0.7921266555786133,
 0.7921266555786133,
 0.7921265363693237,
 0.7921265959739685,
 0.7921266555786133,
 0.7921266555786133,
 0.7921267747879028,
 0.7921266555786133,
 0.7921265959739685]

In [19]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.7921266436576844 ± 5.840038639982171e-08, Weighted Accuracy: 0.8736116707324981 ± 7.077659420393656e-08, Kendall's Tau: 0.5794762611746573 ± 1.4764874380405343e-07
